# Lecture 10 — From script to dependable pipeline

> *"A scrape that runs once is a demo. A scrape that runs every Tuesday at 06:00 for two years is engineering."*

This is the capstone. The previous nine lectures handed you the parts: HTTP, parsing, JS rendering, async politeness, APIs, anti-bot judgement, LLM extraction. Now we'll bolt them together into something you'd be willing to leave running unattended — not glamorous, just *boring* and *reliable*.

The core insight: 90% of what makes a scrape "production grade" has nothing to do with scraping. It's the same boring engineering that any backend job needs — idempotency, dedup, observability, and a graceful failure mode.

## 1. The shape of a real pipeline

A dependable scrape has five layers, in order of how often you'll touch them:

```
[scheduler]   (cron / systemd timer)            — touched once a quarter
     ↓
[orchestrator] (the entrypoint script)          — touched when you change *what* runs
     ↓
[fetcher]      (httpx, with cache + backoff)    — touched when the site changes
     ↓
[extractor]    (selectors and/or LLM)           — touched when the schema drifts
     ↓
[store]        (sqlite, postgres, files)        — almost never touched
```

Build it bottom-up. Get the store right first — your data outlives your code. Then write the extractor against fixtures. Then the fetcher. Then wire it all together. The scheduler is the last thing you set up, not the first.

## 2. The store: SQLite is enough

For scrapes producing up to a few million rows, a single SQLite file is genuinely the right answer. It is faster than you think, requires zero ops, and lets you `scp` your entire dataset around as one file.

A template schema for a generic "things scraped from URLs" pipeline:

In [ ]:
import sqlite3
from pathlib import Path

DB = Path("scrape.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS pages (
    url           TEXT PRIMARY KEY,
    fetched_at    TEXT NOT NULL,
    status        INTEGER NOT NULL,
    etag          TEXT,
    last_modified TEXT,
    body_sha256   TEXT,
    body          BLOB
);

CREATE TABLE IF NOT EXISTS items (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    source_url      TEXT NOT NULL REFERENCES pages(url),
    extracted_at    TEXT NOT NULL,
    schema_version  INTEGER NOT NULL,
    payload_json    TEXT NOT NULL,
    UNIQUE (source_url, schema_version)
);

CREATE INDEX IF NOT EXISTS items_extracted ON items(extracted_at);
"""

def init_db():
    with sqlite3.connect(DB) as cx:
        cx.executescript(SCHEMA)


Two tables, two ideas:

- **`pages`** is the *raw* layer. One row per URL you've ever fetched. The full body is stored — it's cheap, and you'll be grateful when you need to re-extract with a new schema without re-hitting the site.
- **`items`** is the *extracted* layer. The `(source_url, schema_version)` UNIQUE constraint means re-running extraction is safely idempotent: same URL + same schema version → at most one row.

If this grows beyond what SQLite handles comfortably, you'll know — and the migration to Postgres is the same shape with different DDL.

## 3. The fetcher: cache, backoff, identify

The fetcher is the only layer that talks to the outside world, so it carries the politeness obligations from lecture 06 and the dignity rules from lecture 08.

In [ ]:
import asyncio
import hashlib
from datetime import datetime, timezone
import httpx
from tenacity import AsyncRetrying, stop_after_attempt, wait_exponential, retry_if_exception_type

UA = "my-research-bot/0.3 (+https://example.org/bot; mailto:me@example.org)"

async def fetch(client: httpx.AsyncClient, url: str, conn: sqlite3.Connection) -> bytes | None:
    # Conditional GET if we've seen this URL before.
    row = conn.execute(
        "SELECT etag, last_modified, body FROM pages WHERE url = ?", (url,)
    ).fetchone()
    headers = {"User-Agent": UA}
    if row and row[0]:
        headers["If-None-Match"] = row[0]
    if row and row[1]:
        headers["If-Modified-Since"] = row[1]

    async for attempt in AsyncRetrying(
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        retry=retry_if_exception_type((httpx.TransportError, httpx.HTTPStatusError)),
        reraise=True,
    ):
        with attempt:
            r = await client.get(url, headers=headers, timeout=30)
            if r.status_code == 304 and row:
                return row[2]  # unchanged; reuse cached body
            r.raise_for_status()
            body = r.content
            conn.execute(
                "INSERT OR REPLACE INTO pages "
                "(url, fetched_at, status, etag, last_modified, body_sha256, body) "
                "VALUES (?, ?, ?, ?, ?, ?, ?)",
                (
                    url,
                    datetime.now(timezone.utc).isoformat(),
                    r.status_code,
                    r.headers.get("ETag"),
                    r.headers.get("Last-Modified"),
                    hashlib.sha256(body).hexdigest(),
                    body,
                ),
            )
            conn.commit()
            return body


Three things doing real work here:

- **Conditional GET** with `ETag` / `If-Modified-Since`. On a re-run, most pages return `304 Not Modified` — you spend ~0 bandwidth and the origin server thanks you.
- **`tenacity`** for backoff. Network blips, transient `5xx`, brief rate limits — they all get a few exponential retries before you actually fail.
- **Honest `User-Agent`** with a contact URL. Lecture 08 made the case; this is what it looks like in practice.

## 4. The orchestrator: small, dumb, scheduled

The entrypoint should fit on one screen. If it doesn't, you're hiding logic in it that belongs in the layers below.

In [ ]:
async def run_once(urls: list[str]) -> None:
    init_db()
    limits = httpx.Limits(max_connections=8)  # be polite
    async with httpx.AsyncClient(limits=limits, http2=True) as client:
        with sqlite3.connect(DB) as conn:
            sem = asyncio.Semaphore(4)

            async def one(url: str) -> None:
                async with sem:
                    body = await fetch(client, url, conn)
                    if body is None:
                        return
                    item = extract(body, url)  # your extractor (selectors or LLM)
                    if item is None:
                        return
                    conn.execute(
                        "INSERT OR REPLACE INTO items "
                        "(source_url, extracted_at, schema_version, payload_json) "
                        "VALUES (?, ?, ?, ?)",
                        (
                            url,
                            datetime.now(timezone.utc).isoformat(),
                            SCHEMA_VERSION,
                            json.dumps(item),
                        ),
                    )
                    conn.commit()

            await asyncio.gather(*(one(u) for u in urls))


Scheduling is then just:

```cron
# m h dom mon dow command
 17 6  *  *  *  cd /opt/scrape && /usr/bin/python -m scrape.run >> log.txt 2>&1
```

The `17` (not `00`) is deliberate — *don't* run on the top of the hour. Every other cron-driven scrape on the planet does, and the sites you depend on hate it. Pick a random minute and stick with it.

## 5. Observability: the one log line that matters

You don't need Datadog. You need *one* structured log line per page, plus a summary at the end. Future-you needs to be able to grep for "why didn't we pick up Acme yesterday?" and get an answer in five seconds.

Minimum viable observability:

- One line per fetch: `{url, status, bytes, ms, cache_hit}`.
- One summary line per run: `{started_at, finished_at, urls_total, urls_ok, urls_changed, urls_failed}`.
- A non-zero exit code if more than ~5% of URLs failed. Cron will email you. Cron's email is the cheapest alerting system ever invented and it works.

Upgrade to a real metrics backend when (and only when) you have something worth charting.

## 6. When to graduate to a framework

Everything above fits in roughly 200 lines of Python. That's deliberate — it's all auditable, all yours.

Reach for **Scrapy** when:

- You're crawling *discoverable* sites (following links recursively, not hitting a known URL list).
- You have many spiders sharing infrastructure.
- You need pluggable middlewares (proxies, fingerprinting, sessions) more than you need control.

Reach for **Airflow / Prefect / Dagster** when:

- The scrape is one node in a larger DAG.
- You need sophisticated retries, backfills, or per-task observability.
- More than one engineer is touching the schedule.

For a single-developer, single-purpose scrape, *don't*. The frameworks earn their weight at scale and become weight when there's no scale to justify them.

## 7. The discipline list

A running checklist of habits that separate scrapes that survive from scrapes that rot:

- **Store raw, extract derived.** Always keep the original bytes. Schemas change; sources don't replay.
- **Schema-version your extracted data.** Bump the version when the shape changes. Never silently mix.
- **Idempotent re-runs.** Running the pipeline twice in a row should produce the same DB state, not double-insert.
- **Pin your dependencies.** A `requirements.txt` with exact versions. `playwright install` writes a browser version too — record it.
- **Test against fixtures, not the live site.** Save 5 representative pages as `.html` files; assert your extractor against those.
- **Document the why.** A `README` paragraph for each spider: who asked, what's the data used for, when can it be turned off.
- **Have a kill switch.** A single env var or config flag that disables the scrape without a code change.
- **Quarterly review.** Open the project, run the tests, read the logs. Things rot quietly otherwise.

## 8. What you've actually learned

Look back at lectures 00–10 as a single arc:

- 00–01 — what the web *is*, and what scraping it means.
- 02–04 — fetching and parsing static content.
- 05–06 — dealing with JavaScript and being polite at scale.
- 07–08 — when not to scrape: APIs first, anti-bot reframed.
- 09 — when selectors aren't the right tool.
- 10 — making it dependable.

The loud, fun parts are 03–05. The parts that decide whether a project ships and survives are 01, 06, 07, 08, and 10. If a year from now you only remember one thing from this course, make it: *the technical part is the easy part.*

## Recap

- A dependable pipeline has five layers — scheduler, orchestrator, fetcher, extractor, store. Build them bottom-up.
- SQLite is the right store for nearly everything you'll build solo. Two tables: raw `pages`, derived `items`.
- The fetcher does conditional GETs, exponential backoff, and identifies itself honestly.
- The orchestrator should fit on one screen.
- One structured log line per page, one summary per run, cron's email for alerting. Upgrade only when there's a real reason.
- Reach for Scrapy / Airflow when scale or team-size *demands* it; otherwise stay small.
- The hardest part of long-running scrapes is discipline, not code: store raw, version schemas, test against fixtures, review quarterly.

## Exercises

1. Take a small set of URLs (5–20) you actually care about. Implement the schema, fetcher, and orchestrator above end-to-end. Run it twice. Confirm the second run is idempotent (no duplicate `items` rows; most fetches return 304).
2. Save 3 pages from your target site as `.html` fixtures. Write a test that runs your extractor against each and asserts the expected output. Now intentionally break a selector — does the test catch it?
3. Schedule the pipeline to run hourly via cron (or a systemd timer). Let it run for 24 hours. Read the logs. What surprised you?
4. Pick one of the 8 items in §7's discipline list that you currently *don't* do. Add it. Notice whether it pays off in the next week.

## Closing

That's the course.

The parts the internet tells you to obsess over (rotating proxies, headless browser farms, captcha solvers) are mostly the wrong fight. The parts almost nobody writes about (storing raw, versioning schemas, picking minute `:17`, replying to the operator's email) are what actually keep a scrape alive for years.

Go build something small, dependable, and honest. Good luck.